In [1]:
from pathlib import Path 
print(Path.cwd())

C:\Users\HP\Desktop\keren\my_project


In [3]:
from pathlib import Path 
raw_folder = Path("data/raw") 
csv_files = list(raw_folder.glob("*.csv")) 
print("Number of CSV files:", len(csv_files)) 
for file in csv_files: 
    print(file.name) 

Number of CSV files: 12
branches.csv
customers.csv
inventory_master.csv
invoices.csv
payments.csv
products.csv
purchase_orders_header.csv
purchase_orders_lines.csv
sales_orders_header.csv
sales_orders_lines.csv
stock_ledger.csv
suppliers.csv


In [4]:
import duckdb 
con = duckdb.connect("warehouse.duckdb") 
print("Database created successfully!")

Database created successfully!


In [11]:
from pathlib import Path 
import duckdb 
 
# Open the database 
con = duckdb.connect("warehouse.duckdb") 
 
# Locate the folder containing your CSV files 
raw_folder = Path("data/raw") 
 
# Find all CSV files 
csv_files = sorted(raw_folder.glob("*.csv")) 
 
# Check the number of files 
if len(csv_files) != 12: 
    raise ValueError( f"Expected 12 CSV files, but found {len(csv_files)}. " 
        "Check the data/raw folder before continuing." 
    ) 
 
# Load each CSV file into DuckDB 
for file in csv_files: 
    table_name = file.stem 
 
    # Quote the table name safely for SQL 
    safe_table_name = '"' + table_name.replace('"', '""') + '"' 
 
    # Import the CSV data into a database table 
    con.execute( 
        f""" 
        CREATE OR REPLACE TABLE {safe_table_name} AS 
        SELECT * 
        FROM read_csv_auto(?) 
        """, 
        [str(file)] 
    ) 
 
    print(f"Loaded: {file.name} -> {table_name}") 
 
print("All 12 CSV files have been loaded successfully!")

Loaded: branches.csv -> branches
Loaded: customers.csv -> customers
Loaded: inventory_master.csv -> inventory_master
Loaded: invoices.csv -> invoices
Loaded: payments.csv -> payments
Loaded: products.csv -> products
Loaded: purchase_orders_header.csv -> purchase_orders_header
Loaded: purchase_orders_lines.csv -> purchase_orders_lines
Loaded: sales_orders_header.csv -> sales_orders_header
Loaded: sales_orders_lines.csv -> sales_orders_lines
Loaded: stock_ledger.csv -> stock_ledger
Loaded: suppliers.csv -> suppliers
All 12 CSV files have been loaded successfully!


In [12]:
tables = con.execute("SHOW TABLES").fetchall() 
 
for table in tables: 
    print(table[0])

branches
customers
inventory_master
invoices
payments
products
purchase_orders_header
purchase_orders_lines
sales_orders_header
sales_orders_lines
stock_ledger
suppliers


In [13]:
table_count = len(con.execute("SHOW TABLES").fetchall()) 
 
print("Total tables:", table_count)

Total tables: 12


In [14]:
tables = con.execute("SHOW TABLES").fetchall()
for (table_name,) in tables: 
    safe_name = '"' + table_name.replace('"', '""') + '"' 
 
    row_count = con.execute( 
        f"SELECT COUNT(*) FROM {safe_name}" 
    ).fetchone()[0] 
 
    print(f"{table_name}: {row_count} rows")

branches: 6 rows
customers: 500 rows
inventory_master: 180 rows
invoices: 18033 rows
payments: 19257 rows
products: 30 rows
purchase_orders_header: 24000 rows
purchase_orders_lines: 155495 rows
sales_orders_header: 20000 rows
sales_orders_lines: 130402 rows
stock_ledger: 237230 rows
suppliers: 8 rows


In [15]:
con.execute( 
    "SELECT * FROM customers LIMIT 5" 
).df()

,customer_id,customer_type,industry_segment,city,state,pincode,region,branch_id,credit_limit,current_balance,payment_terms,customer_since,last_purchase_date,total_purchase_value,customer_rating
0,C0001,Retail,Construction,Kolkata,West Bengal,991035,East,KOL001,45896,43284,Advance,2024-04-10,2024-04-25,3892818,4
1,C0002,Fleet Owner,Manufacturing,Hyderabad,Telangana,464589,South,HYD001,531506,426735,Advance,2022-01-19,2024-12-04,3168730,2
2,C0003,Government,Logistics,Surat,Gujarat,202475,West,DEL001,1306933,733495,Net 30,2017-02-14,2019-06-11,3584708,2
3,C0004,Fleet Owner,Logistics,Chennai,Tamil Nadu,576213,South,CHN001,506336,250255,Net 30,2017-12-20,2021-12-26,3810635,3
4,C0005,Dealer,Construction,Ahmedabad,Gujarat,965661,West,AHM001,380783,183927,Net 30,2023-07-27,2024-02-17,1381374,2


In [16]:
con.execute( 
    "DESCRIBE customers" 
).df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,customer_type,VARCHAR,YES,None,None,None
2,industry_segment,VARCHAR,YES,None,None,None
3,city,VARCHAR,YES,None,None,None
4,state,VARCHAR,YES,None,None,None
5,pincode,BIGINT,YES,None,None,None
6,region,VARCHAR,YES,None,None,None
7,branch_id,VARCHAR,YES,None,None,None
8,credit_limit,BIGINT,YES,None,None,None
9,current_balance,BIGINT,YES,None,None,None


In [18]:
from pathlib import Path 
db_path = Path("warehouse.duckdb") 
print("Database exists:", db_path.exists()) 
if db_path.exists(): 
    print("Database size:", db_path.stat().st_size, 
"bytes")

Database exists: True
Database size: 12070912 bytes


In [19]:
con.close()